In [ ]:
import re
from unstructured.partition.pdf import partition_pdf
#表を読み込むサンプル
# ---------------------------------------------------------
# 1. 語の途中の改行だけ削除する（段落改行は維持）
# ---------------------------------------------------------
def fix_inline_linebreaks(text):
    lines = text.split("\n")
    fixed = []

    for line in lines:
        line = line.rstrip()

        # 前の行が句点で終わらない → 語の途中の改行と判断して結合
        if fixed and not fixed[-1].endswith("。"):
            fixed[-1] = fixed[-1] + line  # 改行削除して結合
        else:
            fixed.append(line)

    return "\n".join(fixed)


# ---------------------------------------------------------
# 2. ページの最後の行が「。」で終わるまで次ページを結合
# ---------------------------------------------------------
def merge_until_period(page_texts):
    merged_pages = []
    i = 0

    while i < len(page_texts):
        # 語の途中改行を修正（段落改行は維持）
        current_text = fix_inline_linebreaks(page_texts[i]["text"])
        lines = current_text.split("\n")

        last_line = lines[-1].rstrip()

        # 句点が付くまで次ページを結合
        while not last_line.endswith("。") and i + 1 < len(page_texts):
            next_text = fix_inline_linebreaks(page_texts[i + 1]["text"])
            next_lines = next_text.split("\n")

            if next_lines:
                # ★ 結合時は改行を削除して連結（あなたの要求）
                lines[-1] = last_line + next_lines[0]

                next_lines.pop(0)
                lines.extend(next_lines)

                last_line = lines[-1].rstrip()

            i += 1

        merged_pages.append("\n".join(lines))
        i += 1

    return merged_pages


# ---------------------------------------------------------
# 3. PDF → テキスト抽出 → PageBreak でページ分割
# ---------------------------------------------------------
elements = partition_pdf("sample.pdf")
full_text = "\n".join(el.text for el in elements)

pages_raw = re.split(r"<!--\s*PageBreak\s*-->", full_text)
page_texts = [{"page": idx, "text": page} for idx, page in enumerate(pages_raw, start=1)]

# ---------------------------------------------------------
# 4. 実行
# ---------------------------------------------------------
merged = merge_until_period(page_texts)

# ---------------------------------------------------------
# 5. 出力
# ---------------------------------------------------------
for page in merged:
    print("---- PAGE ----")
    print(page)


In [2]:
import re
from unstructured.partition.pdf import partition_pdf

# ---------------------------------------------------------
# 1. 不要ヘッダを削除する
# ---------------------------------------------------------
def remove_headers(text):
    # ロゴ（figure）
    text = re.sub(r"<figure>[\s\S]*?</figure>", "", text)

    # コメント版 PRESS RELEASE
    text = re.sub(r"<!--\s*PageHeader=\"PRESS RELEASE\"\s*-->", "", text)

    # テキスト版 PRESS RELEASE
    text = re.sub(r"PRESS RELEASE", "", text)

    # ★ 「各 位」〜「電話番号」までを削除（本文が続いていても削除）
    text = re.sub(
        r"各 位.*?電話番号",   # 「各 位」〜「電話番号」まで
        "",
        text,
        flags=re.DOTALL
    )

    # 日付（例：2021 年1月 12日）
    text = re.sub(r"\d{4}\s*年\s*\d+\s*月\s*\d+\s*日", "", text)

    return text


# ---------------------------------------------------------
# 2. 語の途中の改行だけ削除（段落改行は維持）
# ---------------------------------------------------------
def fix_inline_linebreaks(text):
    lines = text.split("\n")
    fixed = []

    for line in lines:
        line = line.rstrip()

        # 前の行が句点で終わらない → 語の途中の改行と判断して結合
        if fixed and not fixed[-1].endswith("。"):
            fixed[-1] = fixed[-1] + line  # 改行削除して結合
        else:
            fixed.append(line)

    return "\n".join(fixed)


# ---------------------------------------------------------
# 3. ページの最後が「。」で終わるまで次ページを結合
# ---------------------------------------------------------
def merge_until_period(page_texts):
    merged_pages = []
    i = 0

    while i < len(page_texts):
        current_text = fix_inline_linebreaks(page_texts[i]["text"])
        lines = current_text.split("\n")

        last_line = lines[-1].rstrip()

        while not last_line.endswith("。") and i + 1 < len(page_texts):
            next_text = fix_inline_linebreaks(page_texts[i + 1]["text"])
            next_lines = next_text.split("\n")

            if next_lines:
                # 結合時は改行削除
                lines[-1] = last_line + next_lines[0]

                next_lines.pop(0)
                lines.extend(next_lines)

                last_line = lines[-1].rstrip()

            i += 1

        merged_pages.append("\n".join(lines))
        i += 1

    return merged_pages


# ---------------------------------------------------------
# 4. PDF → テキスト抽出 → PageBreak でページ分割
# ---------------------------------------------------------
elements = partition_pdf("sample.pdf")
full_text = "\n".join(el.text for el in elements)

# 不要ヘッダ削除
full_text = remove_headers(full_text)

# PageBreak でページ分割
pages_raw = re.split(r"<!--\s*PageBreak\s*-->", full_text)

page_texts = [{"page": idx, "text": page} for idx, page in enumerate(pages_raw, start=1)]

# ---------------------------------------------------------
# 5. 実行
# ---------------------------------------------------------
merged = merge_until_period(page_texts)

# ---------------------------------------------------------
# 6. 出力
# ---------------------------------------------------------
for page in merged:
    print("---- PAGE ----")
    print(page)


C:\Users\hiros\anaconda3\envs\unstructured_practice\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.


---- PAGE ----
®@@ sosel HEPTARES20224F10A48 | 2-H )L—PRASE (A—F : 4565)im SIAKRBAL S—EWT /L-FRASHELUTOF SHAR, BMLT BHT IL—TD) (SBT ORB RO — ARATE ACML CUES. ABS, eRERe BH CLELEO CHY. AiO RADE PRAT ENT AELTHENSNEEO CLHVECA,. ABMS, ZHAO RAGE BH, MBER. ELISHEOI—AISDE CHMENKED CLHVEKA. Ek, HE HICKOA (MEE OME KISS OBRORMARKT SCLeEBRMLCH OF, ONO AMES O PAEKISBASHRT SLONBACLHVECA.KRENOMRL, BN CHISARCLHVECA. —MOGRL, AHIRMRDSAFEN TORS. HHT IL—FIX, ARMHSLERRHICH DOT. ABAICGEN CL STARRO EELE, AEE, KISSES EICALC-ORRBERISAMEES . SRO ERE. AEE. ERISA SONS CEHVECA. BHT ILFIX, HLURB LU ELILHKOWRSICROLANt CKBH eRRT SREELILRBARVECA. Ek, BHAI —TFid, MIA ELILRSS BM OBBAAICCE, BCORSICSYARHONBEEBODEA CHA. (IE. EKISRA CERT. ABAIIIL, 19334 O REEF AEO LIV aV27A(MELEC) BLV1934FO KBREF MS PAOVIVAV21E (MELSC) CEBEN CWS! RF MILT Sac IMBSENTIET. ECS, GTO]. PATO] SRT Sl HHS). BSI. RRtOl. PE CHS]., UAT REEMNHOI. SAUCNOLMRORHIL, HAF BITES Sack CHOCCERLUWET,. ABHICBEN TWOMAEOSRAAOT ATOMS, HROREICES SSA7 I LFOMBKR, SRAM

In [3]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf("sample.pdf")
for el in elements:
    print(type(el), el.text[:80])


No languages specified, defaulting to English.


<class 'unstructured.documents.elements.Title'> ®@@ sosel HEPTARES
<class 'unstructured.documents.elements.Text'> 20224F10A48 | 2-H )L—PRASE (A—F : 4565)
<class 'unstructured.documents.elements.NarrativeText'> im SIA
<class 'unstructured.documents.elements.NarrativeText'> KRBAL S—EWT /L-FRASHELUTOF SHAR, BMLT BHT IL—TD) (SBT ORB RO — ARATE ACML CUES. 
<class 'unstructured.documents.elements.NarrativeText'> KRENOMRL, BN CHISARCLHVECA. —MOGRL, AHIRMRDSAFEN TORS. HHT IL—FIX, ARMHSLERRHICH
<class 'unstructured.documents.elements.NarrativeText'> KREHEBLUTCON RISES AR CHU, SHUT )L—FOSHIKOSRMASSSGOCLE, SMELIS-MEBR, GH. TOMO CA
<class 'unstructured.documents.elements.NarrativeText'> ABRIL, ZEGAAPHBET—-AFMEEN TIES. RAGICZEN Tl SIEGAAPH BT —AIK, IFRSICGED CHR ENKM
<class 'unstructured.documents.elements.Text'> ABHOIBSHEE IE, 20187F1A18 KY RIO HARICRIL CIES 4EO4A18 DO HBFO3F318KCO12HA MS £U
<class 'unstructured.documents.elements.Text'> lSosei Heptares IX, RA EEAFAN S| ATIC LIGLTWSE—EWT L—-PFRASH

In [ ]:
import shutil
print(shutil.which("pdfinfo"))

In [5]:
import subprocess

try:
    subprocess.run(["pdfinfo", "-h"], check=True)
    print("pdfinfo OK")
except Exception as e:
    print("pdfinfo NG", e)


pdfinfo OK


In [1]:
import shutil
print(shutil.which("tesseract"))


C:\Program Files\Tesseract-OCR\tesseract.EXE
